In [1]:
import sys, pathlib

try:
    root = pathlib.Path(__vsc_ipynb_file__).resolve().parent.parent
except NameError:
    p = pathlib.Path().resolve()
    root = p.parent if p.name == "notebooks" else p

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.agent import HelpDeskAgent
from src.configs import Settings
from src.tools.search_xyz_manual import search_xyz_manual
from src.tools.search_xyz_qa import (
    search_xyz_qa,
)

settings = Settings()


In [2]:
agent = HelpDeskAgent(
    settings=settings,
    tools=[search_xyz_manual, search_xyz_qa],
)

In [3]:
question = """
お世話になっております。

現在、XYZシステムの利用を検討しており、以下の2点についてご教示いただければと存じます。

1. パスワードに利用可能な文字の制限について
当該システムにてパスワードを設定する際、使用可能な文字の範囲（例：英数字、記号、文字数制限など）について詳しい情報をいただけますでしょうか。安全かつシステムでの認証エラーを防ぐため、具体的な仕様を確認したいと考えております。

2. 最新リリースの取得方法について
最新のアップデート情報をどのように確認・取得できるかについてもお教えいただけますと幸いです。

お忙しいところ恐縮ですが、ご対応のほどよろしくお願い申し上げます。
"""

# question = """
# お世話になっております。

# 現在、XYZシステムを利用を検討しており、以下の点についてご教示いただければと存じます。

# 1. 特定のプロジェクトに対してのみ通知を制限する方法について

# 2. パスワードに利用可能な文字の制限について
# 当該システムにてパスワードを設定する際、使用可能な文字の範囲（例：英数字、記号、文字数制限など）について詳しい情報をいただけますでしょうか。安全かつシステムでの認証エラーを防ぐため、具体的な仕様を確認したいと考えております。

# お忙しいところ恐縮ですが、ご対応のほどよろしくお願い申し上げます。

# """

## 計画ステップ

In [4]:
input_data = {"question": question}

plan_result = agent.create_plan(state=input_data)

2026-04-26 11:49:58,146 INFO 🚀 Starting plan generation process...
2026-04-26 11:49:58,147 INFO Sending request to OpenAI...
2026-04-26 11:50:00,710 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-26 11:50:00,729 INFO ✅ Successfully received response from OpenAI.
2026-04-26 11:50:00,729 INFO Plan generation complete!


In [5]:
plan_result["plan"]

['XYZシステムのパスワード設定における使用可能な文字の種類（英数字、記号など）について調べる',
 'XYZシステムのパスワード設定における文字数制限について調べる',
 'XYZシステムの最新リリース情報を確認する方法について調べる',
 'XYZシステムの最新リリースを取得する方法について調べる']

## ツール選択ステップ

In [6]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "is_completed": False,
}

### データの確認

In [8]:
input_data

{'question': '\nお世話になっております。\n\n現在、XYZシステムの利用を検討しており、以下の2点についてご教示いただければと存じます。\n\n1. パスワードに利用可能な文字の制限について\n当該システムにてパスワードを設定する際、使用可能な文字の範囲（例：英数字、記号、文字数制限など）について詳しい情報をいただけますでしょうか。安全かつシステムでの認証エラーを防ぐため、具体的な仕様を確認したいと考えております。\n\n2. 最新リリースの取得方法について\n最新のアップデート情報をどのように確認・取得できるかについてもお教えいただけますと幸いです。\n\nお忙しいところ恐縮ですが、ご対応のほどよろしくお願い申し上げます。\n',
 'plan': ['XYZシステムのパスワード設定における使用可能な文字の種類（英数字、記号など）について調べる',
  'XYZシステムのパスワード設定における文字数制限について調べる',
  'XYZシステムの最新リリース情報を確認する方法について調べる',
  'XYZシステムの最新リリースを取得する方法について調べる'],
 'subtask': 'XYZシステムのパスワード設定における使用可能な文字の種類（英数字、記号など）について調べる',
 'challenge_count': 0,
 'is_completed': False}

In [9]:
plan_result

{'plan': ['XYZシステムのパスワード設定における使用可能な文字の種類（英数字、記号など）について調べる',
  'XYZシステムのパスワード設定における文字数制限について調べる',
  'XYZシステムの最新リリース情報を確認する方法について調べる',
  'XYZシステムの最新リリースを取得する方法について調べる']}

In [10]:
select_tool_result = agent.select_tools(state=input_data)

2026-04-26 12:04:13,253 INFO 🚀 Starting tool selection process...
2026-04-26 12:04:13,366 INFO Sending request to OpenAI...
2026-04-26 12:04:17,332 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-26 12:04:17,336 INFO ✅ Successfully received response from OpenAI.
2026-04-26 12:04:17,337 INFO Tool selection complete!


In [11]:
select_tool_result

{'messages': [{'role': 'system',
   'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
  {'role': 'user',
   'content': "\nユーザーの元の質問: \nお世話になっております。\

In [12]:
select_tool_result["messages"][-1]

{'role': 'assistant',
 'tool_calls': [{'id': 'call_eivMP6rk3dBwyKOUe79O0ya0',
   'function': {'arguments': '{"keywords":"パスワード 使用可能な文字 XYZシステム"}',
    'name': 'search_xyz_manual'},
   'type': 'function'}]}

In [13]:
select_tool_result["messages"]

[{'role': 'system',
  'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
 {'role': 'user',
  'content': "\nユーザーの元の質問: \nお世話になっております。\n\n現在、XYZシステムの利用

## ツール実行ステップ

In [15]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "messages": select_tool_result["messages"],
    "is_completed": False,
}

In [18]:
tool_results = agent.execute_tools(state=input_data)

2026-04-26 12:18:45,668 INFO 🚀 Starting tool execution process...
2026-04-26 12:18:45,672 INFO Searching XYZ manual by keyword: {"keywords":"パスワード 使用可能な文字 XYZシステム"}
2026-04-26 12:18:45,835 INFO POST http://localhost:9200/documents/_search [status:200 duration:0.161s]
2026-04-26 12:18:45,835 INFO Search results: 10 hits
2026-04-26 12:18:45,835 INFO Finished searching XYZ manual by keyword
2026-04-26 12:18:45,836 INFO Tool execution complete!


In [19]:
tool_results["tool_results"][0][0].results

[SearchOutput(file_name='XYZシステム統合ユーザーマニュアル.pdf', content='情報の更新\n定期的に情報を⾒直し、最新の状態を保つこと をお勧めします 。\n5.2 パスワードの定期的な変更\nパスワード変更の⼿順\n「パスワード変更」オプションを選択 し、現在のパス ワードを⼊⼒後、新しい\nパスワードを設定します 。\nパスワードの強度確認\nシステムはパス ワードの強度を⾃動チ ェックし、弱いパス ワードは拒否されま\nす。*1\n*1: パスワードの強度は以下の基準で 評価されます：\n最低8⽂字以上の⻑さ\n⼤⽂字と⼩⽂字の両⽅を含む\n少なくとも1 つの数字を含む\n少なくとも1 つの特殊⽂字（ !@#$%^&* 等）を含む'),
 SearchOutput(file_name='XYZシステム統合ユーザーマニュアル.pdf', content='始するための重要な情報を提供します 。\n2.1 ログイン画⾯へのアクセス\n公式URL へのアクセス\n安全なブラウザを開き、 XYZ システムの公式 URL を⼊⼒します 。\nHTTPS接続の確認\n必ずHTTPS 接続であること を確認し 、フィッシング詐欺を防ぎます 。\n2.2 ユーザー名とパスワードの⼊⼒\nユーザー名の⼊⼒\nシステム管理者から提供された固有のユ ーザー名を⼊⼒します 。\nパスワードの⼊⼒\n強⼒で安全なパス ワードを⼊⼒します 。パスワードは定期的に変更すること をお\n勧めします 。*\n* パスワードを3 回以上間違え るとアカウントがロックされます 。アカウ ントのロッ'),
 SearchOutput(file_name='XYZシステム統合ユーザーマニュアル.pdf', content='ます。\nオンラインヘル プセンター：24時間365 ⽇利⽤可能なヘル プセンター では、シ ス\nテムの使⽤⽅法や⼀般的な問題に関す る問い合わせを受け付けています 。\n定期的なウェビ ナー：新機能の紹介や⾼度な使⽤テクニックを学ぶためのオン\nラインセミナーを開催し ています 。\n2. システムへのアクセスとログイン\nXYZシステムへのア クセスとログ インは、システムを利⽤する上で最初のステッ プで\nす。この章では、安全かつ効率

In [20]:
tool_results

{'messages': [{'role': 'system',
   'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
  {'role': 'user',
   'content': "\nユーザーの元の質問: \nお世話になっております。\

## サブタスク回答

In [21]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "messages": tool_results["messages"],
    "tool_results": tool_results["tool_results"],
    "is_completed": False,
}

In [22]:
subtask_answer = agent.create_subtask_answer(state=input_data)

2026-04-26 12:23:59,632 INFO 🚀 Starting subtask answer creation process...
2026-04-26 12:23:59,633 INFO Sending request to OpenAI...
2026-04-26 12:24:01,502 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-26 12:24:01,505 INFO ✅ Successfully received response from OpenAI.
2026-04-26 12:24:01,506 INFO Subtask answer creation complete!


In [23]:
subtask_answer

{'messages': [{'role': 'system',
   'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
  {'role': 'user',
   'content': "\nユーザーの元の質問: \nお世話になっております。\

In [24]:
print(subtask_answer["subtask_answer"])

XYZシステムのパスワード設定における使用可能な文字の種類について、以下の情報が得られました。

- パスワードは最低8文字以上の長さが必要です。
- 大文字と小文字の両方を含む必要があります。
- 少なくとも1つの数字を含む必要があります。
- 少なくとも1つの特殊文字（例: !@#$%^&* など）を含む必要があります。

これにより、パスワードの強度が自動的にチェックされ、弱いパスワードは拒否されます。


## リフレクション

In [25]:
input_data = {
    "question": question,
    "plan": plan_result["plan"],
    "subtask": plan_result["plan"][0],
    "challenge_count": 0,
    "messages": subtask_answer["messages"],
    "tool_results": tool_results["tool_results"],
    "is_completed": False,
    "subtask_answer": subtask_answer["subtask_answer"],
}

In [26]:
reflection_result = agent.reflect_subtask(state=input_data)

2026-04-26 12:32:47,811 INFO 🚀 Starting reflection process...
2026-04-26 12:32:47,813 INFO Sending request to OpenAI...
2026-04-26 12:32:48,709 INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-26 12:32:48,718 INFO ✅ Successfully received response from OpenAI.
2026-04-26 12:32:48,721 INFO Reflection complete!


In [27]:
reflection_result

{'messages': [{'role': 'system',
   'content': '\nあなたはXYZというシステムの質問応答のためにサブタスク実行を担当するエージェントです。\n回答までの全体の流れは計画立案 → サブタスク実行 [ツール実行 → サブタスク回答 → リフレクション] → 最終回答となります。\nサブタスクはユーザーの質問に回答するために考えられた計画の一つです。\n最終的な回答は全てのサブタスクの結果を組み合わせて別エージェントが作成します。\nあなたは以下の1~3のステップを指示に従ってそれぞれ実行します。各ステップは指示があったら実行し、同時に複数ステップの実行は行わないでください。\nなおリフレクションの結果次第で所定の回数までツール選択・実行を繰り返します。\n\n1. ツール選択・実行\nサブタスク回答のためのツール選択と選択されたツールの実行を行います。\n2回目以降はリフレクションのアドバイスに従って再実行してください。\n\n2. サブタスク回答\nツールの実行結果はあなたしか観測できません。\nツールの実行結果から得られた回答に必要なことは言語化し、最後の回答用エージェントに引き継げるようにしてください。\n例えば、概要を知るサブタスクならば、ツールの実行結果から概要を言語化してください。\n手順を知るサブタスクならば、ツールの実行結果から手順を言語化してください。\n回答できなかった場合は、その旨を言語化してください。\n\n3. リフレクション\nツールの実行結果と回答から、サブタスクに対して正しく回答できているかを評価します。\n回答がわからない、情報が見つからないといった内容の場合は評価をNGにし、やり直すようにしてください。\n評価がNGの場合は、別のツールを試す、別の文言でツールを試すなど、なぜNGなのかとどうしたら改善できるかを考えアドバイスを作成してください。\nアドバイスの内容は過去のアドバイスと計画内の他のサブタスクと重複しないようにしてください。\nアドバイスの内容をもとにツール選択・実行からやり直します。\n評価がOKの場合は、サブタスク回答を終了します。\n\n'},
  {'role': 'user',
   'content': "\nユーザーの元の質問: \nお世話になっております。\

In [28]:
# 最初に選択されたツールを確認
print(reflection_result["messages"][2]["tool_calls"][0]["function"]["name"])

search_xyz_manual


In [29]:
# リフレクション結果の確認
print("is_completed =", reflection_result["reflection_results"][0].is_completed)
print("advice =", reflection_result["reflection_results"][0].advice)

is_completed = True
advice = 
